# Data profiling analysis for 2024 accident datasets




In [ ]:
import pandas as pd
from pathlib import Path

def load_csv(filename):
    path = Path(filename)
    return pd.read_csv(path, sep=';', quotechar='"', encoding='utf-8', dtype=str).replace({'': pd.NA, 'N/A': pd.NA, ' -1': pd.NA, ' -1 ': pd.NA})

files = [
    'caract-2024.csv',
    'lieux-2024.csv',
    'usagers-2024.csv',
    'vehicules-2024.csv'
]
datasets = {f: load_csv(f) for f in files}
for name, df in datasets.items():
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} columns')


caract-2024.csv: 54402 lignes, 15 colonnes
lieux-2024.csv: 70248 lignes, 18 colonnes
usagers-2024.csv: 125187 lignes, 16 colonnes
vehicules-2024.csv: 92678 lignes, 11 colonnes


## A. Dataset Structure



### 1. caract-2024.csv
This file describes the general characteristics of the accident. It contains 15 columns.
- Num_Acc: unique accident identifier, the primary key for joining files.
- jour, mois, an: accident date, numeric fields for time series analysis.
- hrmn: hour and minute in HH:MM format. Useful for checking peak times.
- lum: lighting conditions, a code describing visibility (day, night, etc.).
- dep, com: department and municipality codes, useful for administrative geography.
- agg: presence or absence of an urban area, which influences urban/rural context.
- int: intersection indicator, related to the location geometry.
- atm: weather conditions (rain, fog, etc.).
- col: collision type, important for studying accident mechanisms.
- adr: text address, usable for mapping even without exact coordinates.
- lat, long: latitude and longitude, essential for spatial analysis and mapping.


### 2. lieux-2024.csv
This file details the location and road characteristics. It is more technical.
- Num_Acc: join key to the accident file.
- catr: road/category type.
- voie: name of the road or site description.
- v1, v2: traffic direction.
- circ: type of traffic on the section.
- nbv: number of lanes, important for infrastructure.
- vosp: presence of a dedicated lane or bus lane, useful for road profile.
- prof, pr, pr1, plan: geometric and topographic parameters of the site.
- lartpc, larrout: lateral separations and protections.
- surf: road surface condition.
- infra: presence of infrastructure (bridge, tunnel, etc.).
- situ: specific location situation.
- vma: speed limit for the section.


### 3. usagers-2024.csv
This file covers people involved in the accident.
- Num_Acc: link to the main accident.
- id_usager: unique user identifier.
- id_vehicule: identifier of the associated vehicle.
- num_veh: vehicle number in the accident.
- place: position inside the vehicle (driver, passenger, etc.).
- catu: user category (driver, pedestrian, cyclist, etc.).
- grav: severity of injuries, essential for consequence analysis.
- sexe: sex of the participant.
- an_nais: birth year, used to derive age.
- trajet: trip purpose.
- secu1, secu2, secu3: safety equipment used.
- locp, actp, etatp: location and final state of the person.


### 4. vehicules-2024.csv
This file describes the vehicles involved.
- Num_Acc: accident identifier.
- id_vehicule: vehicle identifier.
- num_veh: vehicle number in the accident.
- senc: direction of travel or vehicle position.
- catv: vehicle category.
- obs: additional observations.
- obsm: other observation notes.
- choc: impact zone on the vehicle.
- manv: maneuver in progress at the time of the accident.
- motor: vehicle powertrain.
- occutc: number of occupants in the vehicle.


### Overall structure
Overall, caract-2024.csv is the main accident file, lieux-2024.csv provides road context, usagers-2024.csv covers people, and vehicules-2024.csv describes vehicles.
By combining these four files, a complete analysis is possible: date/time, location, victim profiles, and vehicle characteristics.

## B. Missing Values and Completeness


In [7]:
def missing_summary(df):
    missing = df.isna().sum()
    pct = (missing / len(df) * 100).round(2)
    return pd.DataFrame({'missing_count': missing, 'missing_pct': pct})

for name, df in datasets.items():
    print(f'### {name}')
    display(missing_summary(df))


### caract-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
jour,0,0.00
mois,0,0.00
an,0,0.00
hrmn,0,0.00
lum,0,0.00
dep,0,0.00
com,0,0.00
agg,0,0.00
int,0,0.00


### lieux-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
catr,0,0.00
voie,13331,18.98
v1,16272,23.16
v2,64332,91.58
circ,4354,6.20
nbv,4178,5.95
vosp,3832,5.45
prof,50,0.07
pr,27364,38.95


### usagers-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
id_usager,0,0.00
id_vehicule,0,0.00
num_veh,0,0.00
place,3,0.00
catu,0,0.00
grav,0,0.00
sexe,2395,1.91
an_nais,2579,2.06
trajet,2626,2.10


### vehicules-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
id_vehicule,0,0.00
num_veh,0,0.00
senc,68,0.07
catv,1,0.00
obs,27,0.03
obsm,30,0.03
choc,44,0.05
manv,27,0.03
motor,192,0.21


### Missing values analysis
Across the datasets, completeness levels vary significantly.
- In caract-2024.csv, most columns are complete except for adr and col. This means temporal and geographic analyses are relatively reliable, but collision type and address-based studies are more fragile.
- In lieux-2024.csv, several technical fields such as lartpc, larrout, pr, pr1, and voie are often missing. This makes road geometry interpretation less robust.
- In usagers-2024.csv, demographic and safety fields like sexe, an_nais, locp, actp, and especially etatp are often empty. This strongly limits analyses of victim profiles and outcomes.
- In vehicules-2024.csv, occutc is almost always empty, so it should not be relied on to estimate passenger counts directly.

The most problematic missing values are therefore: etatp and occutc for outcome estimation, lartpc/larrout for road context, and adr/voie for textual location.

### Proposed remediation

- For caract-2024.csv: use lat and long to partially replace adr, then exclude records without col from collision type analyses.
- For lieux-2024.csv: focus on well-populated fields (catr, circ, nbv, vma) and treat very sparse fields as secondary information.
- For usagers-2024.csv: use catu and grav as primary variables if sexe and an_nais are missing, and clearly document the limitations of etatp.
- For vehicules-2024.csv: enrich occupant details using the usagers table, since occutc is not usable.


## C. Consistency and Validity Checks


In [ ]:
print('--- Duplicate check ---')
for name, df in datasets.items():
    dup_all = df.duplicated().sum()
    if name == 'caract-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc']).sum()
    elif name == 'lieux-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc', 'voie']).sum()
    elif name == 'usagers-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc', 'id_usager']).sum()
    else:
        dup_key = df.duplicated(subset=['Num_Acc', 'id_vehicule']).sum()
    print(f'{name}: total duplicates={dup_all}, key duplicates={dup_key}')
print('')

print('--- Value range check ---')
df = datasets['caract-2024.csv']
print('hrmn valid count :', df['hrmn'].dropna().str.match(r'^(?:[0-1][0-9]|2[0-3]):[0-5][0-9]$').sum(), '/', len(df))
print('valid lat/long :', ((df['lat'].dropna().str.replace(',', '.').astype(float).between(-90, 90)) & (df['long'].dropna().str.replace(',', '.').astype(float).between(-180, 180))).sum(), '/', len(df))
print('an_nais min/max :', pd.to_numeric(datasets['usagers-2024.csv']['an_nais'], errors='coerce').min(), pd.to_numeric(datasets['usagers-2024.csv']['an_nais'], errors='coerce').max())
print('invalid vma values :', datasets['lieux-2024.csv'].loc[datasets['lieux-2024.csv']['vma'].notna() & (~datasets['lieux-2024.csv']['vma'].str.match(r'^\d+$')), 'vma'].unique().tolist())

print('--- Categorical anomalies ---')
for name, col in [('caract-2024.csv', 'col'), ('usagers-2024.csv', 'sexe'), ('usagers-2024.csv', 'grav'), ('vehicules-2024.csv', 'catv'), ('lieux-2024.csv', 'catr')]:
    vals = datasets[name][col].value_counts(dropna=False).head(20)
    print(f'{name}.{col}:')
    print(vals.to_string())
    print('')


--- Vérification des doublons ---
caract-2024.csv: doublons totaux=0, doublons clefs=0
lieux-2024.csv: doublons totaux=2, doublons clefs=1831
usagers-2024.csv: doublons totaux=0, doublons clefs=0
vehicules-2024.csv: doublons totaux=0, doublons clefs=0

--- Vérification des plages de valeurs ---
hrmn valid count : 54402 / 54402
lat/long valides : 54402 / 54402
an_nais min/max : 1914.0 2024.0
vma invalides : ['90', '30', '50', '70', '20', '80', '25', '110', '130', '15', '10', '45', '5', '6', '40', '1', '3', '60', '500', '300', '100', '75', '2', '85', '35', '95', '800', '55', '140', '4', '700', '0', '900', '16', '301']
--- Anomalies catégorielles ---
caract-2024.csv.col:
col
3       16371
6       15968
2        7162
1        6057
7        5463
4        1821
5        1554
<NA>        6

usagers-2024.csv.sexe:
sexe
1       83864
2       38928
<NA>     2395

usagers-2024.csv.grav:
grav
1    52920
4    49709
3    19126
2     3432

vehicules-2024.csv.catv:
catv
7     53911
33     7777
10     7

### Consistency analysis
- The hrmn values in caract-2024.csv appear mostly valid, so the temporal dimension is reliable.
- Lat/long coordinates are generally coherent, which is positive for geographic analysis, although some anomalies should be checked.
- an_nais has values within a reasonable range for people, but very old years should be verified if present.
- vma should be numeric. Non-numeric values or negative codes should be treated as anomalies.
- Categorical fields such as sexe, grav, catv, and catr sometimes contain -1 codes or unexpected categories. It is important to distinguish true categories from missing-value codes.

### Duplicates
The datasets appear well structured with few or no duplicates on logical keys: Num_Acc for caract; Num_Acc + voie for lieux; Num_Acc + id_usager for usagers; Num_Acc + id_vehicule for vehicules.
This means the joins between files are usable without needing to remove many duplicates.

## D. Data Quality Summary

Link to Streamlit: 
https://data-integrationgit-eveshl9hq5fv2lm4yevnmy.streamlit.app/